# Title + Objective (Markdown)

Final “one-time” test evaluation

# Final Evaluation — EcoType Forest Cover Classification

## Objective
This notebook performs the final locked evaluation of the selected tuned model.

### Goals
- Load the best tuned pipeline artifact
- Retrain on the full training data
- Evaluate once on the held-out test set
- Generate confusion matrix and classification report
- Compare CV performance vs test performance for generalization check
- Save:
  - `reports/final/final_report.md`
  - `reports/final/metrics.json`

In [10]:
from pathlib import Path
import json
import joblib
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_validate

# project imports
from src.modeling.metrics import (
    evaluate_classification,
    save_evaluation_outputs,
    save_confusion_matrix_png,
)
from src.modeling.cv import make_stratified_kfold

In [11]:
PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()

CONFIG_DIR = PROJECT_ROOT / "config"
PATHS_FILE = CONFIG_DIR / "paths.yaml"
TRAIN_FILE = CONFIG_DIR / "train.yaml"

with open(PATHS_FILE, "r", encoding="utf-8") as f:
    paths_cfg = yaml.safe_load(f)

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)

ARTIFACTS_TUNING_DIR = PROJECT_ROOT / "artifacts" / "tuning"

DATA_DIR = PROJECT_ROOT / paths_cfg["data"]["processed_dir"]
REPORTS_DIR = PROJECT_ROOT / paths_cfg["artifacts"]["reports_dir"]
FIGURES_DIR = PROJECT_ROOT / paths_cfg["artifacts"]["figures_dir"]
MODELS_DIR = PROJECT_ROOT / paths_cfg["artifacts"]["models_dir"]

FINAL_REPORT_DIR = REPORTS_DIR / "final_evaluation"
FINAL_FIGURES_DIR = FIGURES_DIR / "final_evaluation"
FINAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = train_cfg["data"]["target_col"]

# Load Best Model + Test Data (Code)

Load joblib model

Load X_test, y_test

In [4]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")[TARGET_COL]
y_test = pd.read_csv(DATA_DIR / "y_test.csv")[TARGET_COL]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (116712, 20)
X_test : (29178, 20)
y_train: (116712,)
y_test : (29178,)


## Select Tuned Model Artifact

In [12]:
FINAL_MODEL_NAME = "random_forest"

BEST_ESTIMATOR_PATH = ARTIFACTS_TUNING_DIR / f"{FINAL_MODEL_NAME}_best_estimator.joblib"
BEST_PARAMS_PATH = ARTIFACTS_TUNING_DIR / f"{FINAL_MODEL_NAME}_best_params.yaml"
CV_RESULTS_PATH = ARTIFACTS_TUNING_DIR / f"{FINAL_MODEL_NAME}_cv_results.csv"

print(BEST_ESTIMATOR_PATH)
print(BEST_PARAMS_PATH)
print(CV_RESULTS_PATH)
print(BEST_ESTIMATOR_PATH.exists(), BEST_PARAMS_PATH.exists(), CV_RESULTS_PATH.exists())

F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\artifacts\tuning\random_forest_best_estimator.joblib
F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\artifacts\tuning\random_forest_best_params.yaml
F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\artifacts\tuning\random_forest_cv_results.csv
True True True


In [14]:
best_estimator = joblib.load(BEST_ESTIMATOR_PATH)

with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
    best_info = yaml.safe_load(f)

cv_results_df = pd.read_csv(CV_RESULTS_PATH)

print("Loaded estimator from:", BEST_ESTIMATOR_PATH)
print("Best CV score:", best_info["best_score"])
print("Best params:", best_info["best_params"])

Loaded estimator from: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\artifacts\tuning\random_forest_best_estimator.joblib
Best CV score: 0.8949366792687229
Best params: {'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': None}


# Retrain on full training split

In [15]:
final_pipeline = clone(best_estimator)
final_pipeline.fit(X_train, y_train)

print("Final pipeline retrained on full training data.")

Final pipeline retrained on full training data.


# Predict + Metrics (Code)

y_pred

accuracy

macro f1

print results

In [16]:
y_pred = final_pipeline.predict(X_test)

print("Predictions done.")
print("Unique predicted classes:", sorted(pd.Series(y_pred).unique().tolist()))

Predictions done.
Unique predicted classes: ['Aspen', 'Cottonwood/Willow', 'Douglas-fir', 'Krummholz', 'Lodgepole Pine', 'Ponderosa Pine', 'Spruce/Fir']


# Confusion Matrix (Code)

plot confusion matrix

save figure

In [17]:
labels = sorted(pd.Series(y_train).unique().tolist())

evaluation = evaluate_classification(
    y_true=y_test,
    y_pred=y_pred,
    labels=labels,
)

metrics = evaluation["metrics"]
confusion_matrix_df = evaluation["confusion_matrix_df"]
classification_report_df = evaluation["classification_report_df"]
classification_report_dict = evaluation["classification_report_dict"]

print(metrics)
display(confusion_matrix_df)
display(classification_report_df)

<class 'dict'>
{'Aspen': {'precision': 0.9193825042881647, 'recall': 0.8729641693811075, 'f1-score': 0.8955722639933166, 'support': 614.0}, 'Cottonwood/Willow': {'precision': 0.9493087557603687, 'recall': 0.9537037037037037, 'f1-score': 0.9515011547344111, 'support': 432.0}, 'Douglas-fir': {'precision': 0.8250539956803455, 'recall': 0.8842592592592593, 'f1-score': 0.8536312849162011, 'support': 432.0}, 'Krummholz': {'precision': 0.9349775784753364, 'recall': 0.9652777777777778, 'f1-score': 0.9498861047835991, 'support': 432.0}, 'Lodgepole Pine': {'precision': 0.9670141367985149, 'recall': 0.985543805180945, 'f1-score': 0.976191048218533, 'support': 20614.0}, 'Ponderosa Pine': {'precision': 0.8632075471698113, 'recall': 0.8472222222222222, 'f1-score': 0.8551401869158879, 'support': 432.0}, 'Spruce/Fir': {'precision': 0.9561780374634817, 'recall': 0.8942462230793957, 'f1-score': 0.9241757329125488, 'support': 6222.0}, 'accuracy': 0.9593872095414353, 'macro avg': {'precision': 0.916446079

,pred_Aspen,pred_Cottonwood/Willow,pred_Douglas-fir,pred_Krummholz,pred_Lodgepole Pine,pred_Ponderosa Pine,pred_Spruce/Fir
true_Aspen,536,0,6,0,64,7,1
true_Cottonwood/Willow,0,412,6,0,0,14,0
true_Douglas-fir,3,12,382,0,2,33,0
true_Krummholz,0,0,0,417,3,0,12
true_Lodgepole Pine,30,0,17,5,20316,4,242
true_Ponderosa Pine,4,10,50,0,2,366,0
true_Spruce/Fir,10,0,2,24,622,0,5564


,precision,recall,f1-score,support
Aspen,0.919383,0.872964,0.895572,614.000000
Cottonwood/Willow,0.949309,0.953704,0.951501,432.000000
Douglas-fir,0.825054,0.884259,0.853631,432.000000
Krummholz,0.934978,0.965278,0.949886,432.000000
Lodgepole Pine,0.967014,0.985544,0.976191,20614.000000
Ponderosa Pine,0.863208,0.847222,0.855140,432.000000
Spruce/Fir,0.956178,0.894246,0.924176,6222.000000
accuracy,0.959387,0.959387,0.959387,0.959387
macro avg,0.916446,0.914745,0.915157,29178.000000
weighted avg,0.959326,0.959387,0.959041,29178.000000


# Classification Report (Code)

print report

save report text

# Generalization Check (Markdown)

compare CV score and test score

note if overfitting suspected

In [20]:
cv_best_score = float(best_info["best_score"])
test_f1_macro = float(metrics["f1_macro"])
test_accuracy = float(metrics["accuracy"])
gap = test_f1_macro - cv_best_score

generalization_check = {
    "cv_best_score_f1_macro": cv_best_score,
    "test_f1_macro": test_f1_macro,
    "test_accuracy": test_accuracy,
    "gap_test_minus_cv": gap,
}

generalization_check

{'cv_best_score_f1_macro': 0.8949366792687229,
 'test_f1_macro': 0.9151568252106426,
 'test_accuracy': 0.9593872095414353,
 'gap_test_minus_cv': 0.020220145941919676}

In [21]:
if abs(gap) <= 0.02:
    generalization_note = "Test macro F1 is close to CV macro F1. Generalization looks stable."
elif gap < -0.02:
    generalization_note = "Test macro F1 is lower than CV macro F1. Possible mild overfitting or split variance."
else:
    generalization_note = "Test macro F1 is higher than CV macro F1. Likely favorable split variance."

print(generalization_note)

Test macro F1 is higher than CV macro F1. Likely favorable split variance.


# Save Final Report (Code)

final_metrics.json

final_report.md

In [18]:
final_metrics_json_path = FINAL_REPORT_DIR / "metrics.json"
final_classification_report_csv_path = FINAL_REPORT_DIR / "classification_report.csv"
final_confusion_matrix_csv_path = FINAL_REPORT_DIR / "confusion_matrix.csv"
final_confusion_matrix_png_path = FINAL_FIGURES_DIR / f"{FINAL_MODEL_NAME}_confusion_matrix.png"

save_evaluation_outputs(
    evaluation=evaluation,
    metrics_json_path=final_metrics_json_path,
    classification_report_csv_path=final_classification_report_csv_path,
    confusion_matrix_csv_path=final_confusion_matrix_csv_path,
)

save_confusion_matrix_png(
    y_true=y_test,
    y_pred=y_pred,
    output_path=final_confusion_matrix_png_path,
    labels=labels,
    title=f"Final Confusion Matrix - {FINAL_MODEL_NAME}",
)

print("Saved evaluation outputs.")

Saved evaluation outputs.


In [19]:
final_model_path = MODELS_DIR / f"{FINAL_MODEL_NAME}_final_model.joblib"
joblib.dump(final_pipeline, final_model_path)

print("Saved final model to:", final_model_path)

Saved final model to: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\models\random_forest_final_model.joblib


In [22]:
final_report_md_path = FINAL_REPORT_DIR / "final_report.md"

report_lines = []
report_lines.append("# Final Evaluation Report — EcoType Forest Cover Classification\n")

report_lines.append("## 1. Final Model")
report_lines.append(f"- Selected tuned model: `{FINAL_MODEL_NAME}`")
report_lines.append(f"- Best estimator path: `{BEST_ESTIMATOR_PATH}`")
report_lines.append(f"- Best params path: `{BEST_PARAMS_PATH}`")
report_lines.append(f"- CV results path: `{CV_RESULTS_PATH}`")
report_lines.append(f"- Final refit model path: `{final_model_path}`")
report_lines.append("")

report_lines.append("## 2. Dataset Split")
report_lines.append(f"- X_train shape: `{X_train.shape}`")
report_lines.append(f"- X_test shape: `{X_test.shape}`")
report_lines.append(f"- y_train shape: `{y_train.shape}`")
report_lines.append(f"- y_test shape: `{y_test.shape}`")
report_lines.append("")

report_lines.append("## 3. Final Test Metrics")
for k, v in metrics.items():
    report_lines.append(f"- **{k}**: `{v:.4f}`")
report_lines.append("")

report_lines.append("## 4. Generalization Check")
report_lines.append(f"- CV best macro F1: `{cv_best_score:.4f}`")
report_lines.append(f"- Test macro F1: `{test_f1_macro:.4f}`")
report_lines.append(f"- Test accuracy: `{test_accuracy:.4f}`")
report_lines.append(f"- Gap (test - CV): `{gap:.4f}`")
report_lines.append(f"- Interpretation: {generalization_note}")
report_lines.append("")

report_lines.append("## 5. Best Hyperparameters")
for k, v in best_info["best_params"].items():
    report_lines.append(f"- **{k}**: `{v}`")
report_lines.append("")

report_lines.append("## 6. Classification Report")
report_lines.append(classification_report_df.to_markdown())
report_lines.append("")

report_lines.append("## 7. Saved Outputs")
report_lines.append(f"- Metrics JSON: `{final_metrics_json_path}`")
report_lines.append(f"- Classification report CSV: `{final_classification_report_csv_path}`")
report_lines.append(f"- Confusion matrix CSV: `{final_confusion_matrix_csv_path}`")
report_lines.append(f"- Confusion matrix PNG: `{final_confusion_matrix_png_path}`")

with open(final_report_md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print("Saved final report to:", final_report_md_path)

Saved final report to: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\reports\final_evaluation\final_report.md
